In [ ]:
%pip install networkx matplotlib pandas python-louvain

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 15.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 31.3 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 41.0 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 51.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 26.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 51.3 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 48.8 MB/s  0:00:00
  Created wheel for python-louvain: filename=python_louvain-0.16-py3-none-any.whl size=9459 sha256=237516b3fb265b7472d7331dfebc469d82a18fd8606201b8730d277a874b1fa3
  Stored in directory: /home/codespace/.cache/pip/wheels/40/f1/e3/485b698c520fa0baee1d07897abc7b8d6479b7d199ce96f4af
Successfully built python-louvain
  

In [ ]:
import pandas as pd
import random
from datetime import datetime, timedelta

# 1. Definición estática de Nodos con sus Roles y Comunidades para garantizar consistencia
nodos_info = {
    # Comunidad: GABINETE
    'MIN_PALPATINE': ('POLITICO', 'GABINETE'),
    'ASE_TARKIN': ('ASESOR', 'GABINETE'),
    'ASE_THRAWN': ('ASESOR', 'GABINETE'),
    'POL_VALORUM': ('POLITICO', 'GABINETE'),
    'POL_PADME': ('POLITICO', 'GABINETE'),
    'ASE_PESTAGE': ('ASESOR', 'GABINETE'),
    'ASE_DOOKU': ('ASESOR', 'GABINETE'),
    
    # Comunidad: COMPRAS
    'FUN_KRENNIC': ('FUNCIONARIO', 'COMPRAS'),
    'FUN_HOFF': ('FUNCIONARIO', 'COMPRAS'),
    'FUN_JERJERROD': ('FUNCIONARIO', 'COMPRAS'),
    'FUN_GRIEVOUS': ('FUNCIONARIO', 'COMPRAS'),
    'FUN_OURO': ('FUNCIONARIO', 'COMPRAS'),
    
    # Comunidad: FISCALIZACION
    'FUN_PIETT': ('FUNCIONARIO', 'FISCALIZACION'),
    'FUN_YULAREN': ('FUNCIONARIO', 'FISCALIZACION'),
    'FUN_NEEDA': ('FUNCIONARIO', 'FISCALIZACION'),
    
    # Comunidad: INTERMEDIARIOS (El nexo crucial)
    'INT_BOBA_FETT': ('INTERMEDIARIO', 'INTERMEDIARIOS'), # NODO PUENTE PRINCIPAL
    'INT_CAD_BANE': ('INTERMEDIARIO', 'INTERMEDIARIOS'),
    'INT_GREEDO': ('INTERMEDIARIO', 'INTERMEDIARIOS'),
    'INT_ZAM_WESELL': ('INTERMEDIARIO', 'INTERMEDIARIOS'),
    'CON_FETT_JANGO': ('CONTACTO', 'INTERMEDIARIOS'),
    
    # Comunidad: PROVEEDORES
    'EMP_VADER': ('EMPRESARIO', 'PROVEEDORES'),
    'EMP_XIZOR': ('EMPRESARIO', 'PROVEEDORES'),
    'EMP_GUNRAY': ('EMPRESARIO', 'PROVEEDORES'),
    'EMP_RENN': ('EMPRESARIO', 'PROVEEDORES'),
    'CON_JABBA': ('CONTACTO', 'PROVEEDORES'),
    'CON_LANDO': ('CONTACTO', 'PROVEEDORES'),
    'CON_MAUL': ('CONTACTO', 'PROVEEDORES'),
}

# 2. Listas de categorías permitidas según las especificaciones de la imagen
tipos_relacion_monetaria = ['TRANSFERENCIA', 'ADJUDICACION']
tipos_relacion_no_monetaria = ['REUNION', 'LLAMADA', 'MENSAJE', 'FAVOR', 'ENCUBRIMIENTO', 'PRESION']

comentarios_opciones = [
    "Coordinación de bases de licitación.",
    "Llamada telefónica fuera de horario laboral.",
    "Transferencia de fondos a cuenta externa.",
    "Reunión informal en cafetería céntrica.",
    "Presión indebida para acelerar firma.",
    "Mensaje encriptado detectado por auditoría.",
    "Adjudicación directa sin concurso público.",
    "Favor político solicitado con urgencia.",
    "Encubrimiento de faltas administrativas.",
    "Modificación de contrato a última hora."
]

# Base de fechas coherente
fecha_base = datetime(2026, 1, 1)

aristas = []
aristas_creadas = set()
id_contador = 1

def agregar_arista(origen, destino, tipo=None, riesgo=None, peso=None):
    global id_contador
    if origen == destino or (origen, destino) in aristas_creadas:
        return
    
    rol_orig, com_orig = nodos_info[origen]
    rol_dest, com_dest = nodos_info[destino]
    
    if tipo is None:
        # 70% relaciones normales, 30% financieras si es con proveedores
        if 'PROVEEDORES' in [com_orig, com_dest] or 'INTERMEDIARIOS' in [com_orig, com_dest]:
            tipo = random.choice(tipos_relacion_monetaria + tipos_relacion_no_monetaria)
        else:
            tipo = random.choice(tipos_relacion_no_monetaria)
            
    if peso is None:
        peso = random.randint(1, 5)
        
    if riesgo is None:
        # Mayor probabilidad de riesgo en transacciones o presiones
        riesgo = 1 if tipo in ['TRANSFERENCIA', 'ADJUDICACION', 'ENCUBRIMIENTO', 'PRESION'] and random.random() > 0.4 else 0

    # Cumplir regla: Monto = 0 si no es monetaria
    monto = random.randint(15, 100) if tipo in tipos_relacion_monetaria else 0
    
    fecha = (fecha_base + timedelta(days=random.randint(0, 90))).strftime('%Y-%m-%d')
    comentario = random.choice(comentarios_opciones)
    id_arista = f"E{id_contador:03d}"
    
    aristas.append({
        'id_arista': id_arista,
        'origen': origen,
        'destino': destino,
        'rol_origen': rol_orig,
        'rol_destino': rol_dest,
        'comunidad_origen': com_orig,
        'comunidad_destino': com_dest,
        'tipo_relacion': tipo,
        'peso': peso,
        'monto': monto,
        'fecha': fecha,
        'riesgo': riesgo,
        'comentario': comentario
    })
    
    aristas_creadas.add((origen, destino))
    id_contador += 1

# --- CONSTRUCCIÓN DIRIGIDA DE LA ESTRUCTURA DE LA RED ---

# A. Conexiones del núcleo político (Gabinete e Institución)
politicos = [n for n, info in nodos_info.items() if info[1] == 'GABINETE']
funcionarios_compras = [n for n, info in nodos_info.items() if info[1] == 'COMPRAS']
fiscalizadores = [n for n, info in nodos_info.items() if info[1] == 'FISCALIZACION']
proveedores = [n for n, info in nodos_info.items() if info[1] == 'PROVEEDORES']

for p in politicos:
    for f in funcionarios_compras:
        if random.random() > 0.3:
            agregar_arista(p, f, tipo='PRESION', riesgo=1) # Presión de políticos a compras

for f in funcionarios_compras:
    for fi in fiscalizadores:
        if random.random() > 0.4:
            agregar_arista(f, fi, tipo='ENCUBRIMIENTO', riesgo=random.choice([0, 1]))

# B. El Puente Crítico: INT_BOBA_FETT opera conectando ambos mundos
# Conexiones con el sector gubernamental (Alta centralidad de intermediación)
agregar_arista('MIN_PALPATINE', 'INT_BOBA_FETT', tipo='LLAMADA', peso=4, riesgo=1)
agregar_arista('ASE_TARKIN', 'INT_BOBA_FETT', tipo='REUNION', peso=3, riesgo=0)
agregar_arista('INT_BOBA_FETT', 'FUN_KRENNIC', tipo='FAVOR', peso=5, riesgo=1)
agregar_arista('INT_BOBA_FETT', 'FUN_PIETT', tipo='MENSAJE', peso=2, riesgo=0)

# Conexiones con el sector privado / proveedores
agregar_arista('EMP_VADER', 'INT_BOBA_FETT', tipo='TRANSFERENCIA', peso=5, riesgo=1)
agregar_arista('EMP_XIZOR', 'INT_BOBA_FETT', tipo='TRANSFERENCIA', peso=4, riesgo=1)
agregar_arista('INT_BOBA_FETT', 'CON_JABBA', tipo='REUNION', peso=4, riesgo=1)

# C. Conexiones internas de la red de Proveedores y otros intermediarios
for prov in proveedores:
    for f_comp in funcionarios_compras:
        if random.random() > 0.6:
            agregar_arista(f_comp, prov, tipo='ADJUDICACION', riesgo=1) # Adjudicaciones directas de riesgo
            agregar_arista(prov, f_comp, tipo='TRANSFERENCIA', riesgo=1) # Retornos monetarios

# Rellenar con otras interacciones aleatorias controladas hasta alcanzar la cuota (~85 aristas)
todos_nodos = list(nodos_info.keys())
while len(aristas) < 85:
    u = random.choice(todos_nodos)
    v = random.choice(todos_nodos)
    agregar_arista(u, v)

# 3. Conversión a DataFrame, verificación de cuotas y exportación
df_aristas = pd.DataFrame(aristas)

print("=== VERIFICACIÓN DE REQUERIMIENTOS ===")
print(f"Número de nodos únicos en el dataset: {len(set(df_aristas['origen']).union(set(df_aristas['destino'])))}")
print(f"Número total de aristas generadas: {len(df_aristas)}")
print(f"Número de relaciones con flag_riesgo = 1: {df_aristas['riesgo'].sum()}")
print(f"Roles únicos de origen presentes: {df_aristas['rol_origen'].nunique()}")
print(f"Comunidades únicas de origen presentes: {df_aristas['comunidad_origen'].nunique()}")

# Guardar a archivo CSV en formato UTF-8 tal como se exige
filename = "Luengo_Antonia_T2.csv"
df_aristas.to_csv(filename, index=False, encoding='utf-8')
print(f"\n¡Archivo '{filename}' generado exitosamente en formato UTF-8!")

=== VERIFICACIÓN DE REQUERIMIENTOS ===
Número de nodos únicos en el dataset: 25
Número total de aristas generadas: 85
Número de relaciones con flag_riesgo = 1: 65
Roles únicos de origen presentes: 6
Comunidades únicas de origen presentes: 5

¡Archivo 'Apellido_Nombre_T2.csv' generado exitosamente en formato UTF-8!
